In [ ]:

# Our first Python environment check

print("Hello, Python!")

print("Honeypot AI Classification Project")

# Check the Python version
import sys

print("Python version:", sys.version)


Hello, Python!
Honeypot AI Classification Project
Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]


In [ ]:

# Import the libraries we will need

import os
import json
import pandas as pd

print("Libraries imported successfully!")

# Create a project folder
os.makedirs("honeypot_project/data", exist_ok=True)

print("Project workspace created!")


Libraries imported successfully!
Project workspace created!


In [ ]:

import os

print("Current working directory:")
print(os.getcwd())

print("\nFiles and folders:")
print(os.listdir())

print("\nOur project folder:")
print(os.listdir("honeypot_project"))



Current working directory:
/content

Files and folders:
['.config', 'honeypot_project', 'sample_data']

Our project folder:
['data']


In [ ]:

# TEST GEMINI 3.5 FLASH-LITE CONNECTION

!pip -q install -U google-genai

from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Reply with: Gemini connection successful."
)

print(response.text)



ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.3.0 which is incompatible.
Gemini connection successful.


In [ ]:
from google.colab import files

uploaded = files.upload()


Saving ai_input.csv to ai_input.csv


In [ ]:
import pandas as pd

df = pd.read_csv("ai_input.csv")

print("Dataset successfully loaded!")
print("Total sessions:", len(df))
print("Columns:", list(df.columns))

display(df.head())


Dataset successfully loaded!
Total sessions: 51
Columns: ['session_id', 'login_success', 'login_failed', 'command_count', 'commands', 'event_types']


,session_id,login_success,login_failed,command_count,commands,event_types
0,4f265faea72a,False,False,0,NaN,cowrie.client.kex | cowrie.client.version | co...
1,5b0c375bd6d9,True,False,7,whoami | id | uname -a | pwd | ls -la | cat /e...,cowrie.client.kex | cowrie.client.size | cowri...
2,d5b0e92a0bb7,True,False,5,whoami | id | uname -a | pwd | exit,cowrie.client.kex | cowrie.client.size | cowri...
3,27a1baf6ca1a,False,False,0,NaN,cowrie.client.kex | cowrie.client.version | co...
4,0c279854bc2f,False,False,0,NaN,cowrie.client.kex | cowrie.client.version | co...


In [ ]:

import json
from google.colab import userdata
from google import genai

# Connect to Gemini
client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

# Select the first session
sample = df.iloc[0].fillna("").to_dict()

prompt = f"""
You are assisting with a cybersecurity honeypot research experiment.

Classify the following Cowrie SSH session using its recorded evidence.

Available labels:
1. reconnaissance
2. brute_force
3. malicious_command_attempt
4. insufficient_evidence

Rules:
- reconnaissance: Evidence of system or account discovery.
- brute_force: Evidence of repeated failed authentication attempts.
- malicious_command_attempt: Evidence of attempted suspicious or harmful commands.
- insufficient_evidence: Not enough evidence to confidently assign another label.

Do not assume an attack occurred merely because a connection was established.
Do not invent events or commands.
Explain your decision using the recorded evidence.

Return:
Predicted label:
Reason:
Confidence: High, Medium, or Low.

Session data:
{json.dumps(sample, indent=2)}
"""

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt
)

print(response.text)



Predicted label: insufficient_evidence
Reason: The session logs show only connection establishment, key exchange (kex), client version identification, and immediate session closure (`cowrie.session.connect`, `cowrie.client.kex`, `cowrie.client.version`, `cowrie.session.closed`). There were no login attempts (`login_success` and `login_failed` are both false) and no commands executed (`command_count` is 0). Therefore, there is not enough evidence to classify this as reconnaissance, brute force, or a malicious command attempt.
Confidence: High


In [ ]:

import pandas as pd
import json
import time
from google import genai
from google.colab import userdata

# Connect to Gemini
client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

results = []

for index, row in df.iterrows():

    session = row.fillna("").to_dict()

    prompt = f"""
You are a cybersecurity researcher analysing Cowrie SSH honeypot logs.

Classify this session using only the available evidence.

Allowed labels:
- reconnaissance
- brute_force
- malicious_command_attempt
- insufficient_evidence

Definitions:
- reconnaissance: System or account discovery activity.
- brute_force: Repeated failed authentication attempts.
- malicious_command_attempt: Attempts to execute suspicious or harmful commands.
- insufficient_evidence: Evidence is insufficient to support another label.

Do not invent activity or assume commands succeeded.
Do not confuse connection establishment with an attack.

Return valid JSON only, using these fields:
{{
  "predicted_label": "one allowed label",
  "reason": "brief evidence-based explanation",
  "confidence": "High, Medium, or Low"
}}

Session evidence:
{json.dumps(session, indent=2)}
"""

    try:
        response = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=prompt,
            config={"response_mime_type": "application/json"}
        )

        prediction = json.loads(response.text)

        results.append({
            "session_id": session["session_id"],
            "predicted_label": prediction["predicted_label"],
            "reason": prediction["reason"],
            "confidence": prediction["confidence"]
        })

        print(f"Session {index + 1}/51 completed")

        time.sleep(1)

    except Exception as e:
        print(f"Session {index + 1} failed: {e}")

# Save predictions
predictions = pd.DataFrame(results)

predictions.to_csv(
    "gemini_predictions.csv",
    index=False
)

print("\nCLASSIFICATION COMPLETE")
print("Successful predictions:", len(predictions))
print("Saved to: gemini_predictions.csv")

display(predictions.head())



Session 1/51 completed
Session 2/51 completed
Session 3/51 completed
Session 4/51 completed
Session 5/51 completed
Session 6/51 completed
Session 7/51 completed
Session 8/51 completed
Session 9/51 completed
Session 10/51 completed
Session 11/51 completed
Session 12/51 completed
Session 13/51 completed
Session 14/51 completed
Session 15/51 completed
Session 16/51 completed
Session 17 failed: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-3.5-flash-lite, description: Generate content requests per minute limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 25.128735318s.', 'stat

,session_id,predicted_label,reason,confidence
0,4f265faea72a,insufficient_evidence,The session only shows connection establishmen...,High
1,5b0c375bd6d9,reconnaissance,The attacker successfully logged in and immedi...,High
2,d5b0e92a0bb7,reconnaissance,The attacker successfully logged in and execut...,High
3,27a1baf6ca1a,insufficient_evidence,The session only shows connection establishmen...,High
4,0c279854bc2f,insufficient_evidence,The session only shows connection establishmen...,High


In [ ]:

import pandas as pd
import json
import time
import os
from google import genai
from google.colab import userdata

# Reconnect to Gemini
client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

output_file = "gemini_predictions.csv"

# Preserve existing predictions
if os.path.exists(output_file):
    predictions = pd.read_csv(output_file)
else:
    predictions = pd.DataFrame(columns=[
        "session_id", "predicted_label", "reason", "confidence"
    ])

completed_ids = set(predictions["session_id"].astype(str))

print("Already completed:", len(completed_ids))
print("Remaining:", len(df) - len(completed_ids))

allowed_labels = {
    "reconnaissance",
    "brute_force",
    "malicious_command_attempt",
    "insufficient_evidence"
}

for index, row in df.iterrows():

    session = row.fillna("").to_dict()
    session_id = str(session["session_id"])

    if session_id in completed_ids:
        continue

    prompt = f"""
You are a cybersecurity researcher analysing Cowrie SSH honeypot logs.

Classify this session using only the available evidence.

Allowed labels:
- reconnaissance
- brute_force
- malicious_command_attempt
- insufficient_evidence

Definitions:
- reconnaissance: System or account discovery activity.
- brute_force: Repeated failed authentication attempts.
- malicious_command_attempt: Attempts to execute suspicious or harmful commands.
- insufficient_evidence: Not enough evidence for another category.

Do not invent activity or assume commands succeeded.

Return valid JSON only:
{{
  "predicted_label": "one allowed label",
  "reason": "brief evidence-based explanation",
  "confidence": "High, Medium, or Low"
}}

Session evidence:
{json.dumps(session, indent=2)}
"""

    for attempt in range(4):
        try:
            response = client.models.generate_content(
                model="gemini-3.5-flash-lite",
                contents=prompt,
                config={"response_mime_type": "application/json"}
            )

            result = json.loads(response.text)

            if result["predicted_label"] not in allowed_labels:
                raise ValueError("Unexpected classification label")

            new_row = pd.DataFrame([{
                "session_id": session_id,
                "predicted_label": result["predicted_label"],
                "reason": result["reason"],
                "confidence": result["confidence"]
            }])

            predictions = pd.concat(
                [predictions, new_row], ignore_index=True
            )

            # Save immediately after each successful prediction
            predictions.to_csv(output_file, index=False)
            completed_ids.add(session_id)

            print(f"Session {index + 1}/51 completed")
            time.sleep(5)
            break

        except Exception as e:
            error = str(e)

            if "429" in error or "RESOURCE_EXHAUSTED" in error:
                print("Rate limit reached. Waiting 60 seconds...")
                time.sleep(60)
            else:
                print(f"Session {index + 1} error: {error}")
                break

print("\nRESUMED CLASSIFICATION FINISHED")
print("Successful predictions:", len(predictions))
print("Remaining:", 51 - len(predictions))
print("Saved to:", output_file)



Already completed: 35
Remaining: 16
Session 17/51 completed
Session 18/51 completed
Session 19/51 completed
Session 20/51 completed
Session 22/51 completed
Session 23/51 completed
Session 25/51 completed
Session 26/51 completed
Session 27/51 completed
Session 28/51 completed
Session 45/51 completed
Session 47/51 completed
Session 48/51 completed
Session 49/51 completed
Session 50/51 completed
Session 51/51 completed

RESUMED CLASSIFICATION FINISHED
Successful predictions: 51
Remaining: 0
Saved to: gemini_predictions.csv


In [ ]:
# Check the completed predictions

print("Total predictions:", len(predictions))
print("Unique sessions:", predictions["session_id"].nunique())

print("\nDuplicate session IDs:")
print(predictions[predictions["session_id"].duplicated(keep=False)])

print("\nMissing values:")
print(predictions.isnull().sum())

print("\nPredicted label distribution:")
print(predictions["predicted_label"].value_counts(dropna=False))

allowed_labels = {
    "reconnaissance",
    "brute_force",
    "malicious_command_attempt",
    "insufficient_evidence"
}

print("\nUnexpected labels:")
print(set(predictions["predicted_label"].dropna()) - allowed_labels)

Total predictions: 51
Unique sessions: 51

Duplicate session IDs:
Empty DataFrame
Columns: [session_id, predicted_label, reason, confidence]
Index: []

Missing values:
session_id         0
predicted_label    0
reason             0
confidence         0
dtype: int64

Predicted label distribution:
predicted_label
insufficient_evidence        31
malicious_command_attempt    18
reconnaissance                2
Name: count, dtype: int64

Unexpected labels:
set()


In [ ]:
from google.colab import files
uploaded = files.upload()


Saving human_review.csv to human_review.csv


In [ ]:
import pandas as pd

# Load the human reviewed reference labels
human = pd.read_csv("human_review.csv")

# Load Gemini predictions
predictions = pd.read_csv("gemini_predictions.csv")

# Join both datasets using session_id
comparison = pd.merge(
    human[["session_id", "human_label"]],
    predictions[["session_id", "predicted_label", "confidence"]],
    on="session_id",
    how="inner"
)

print("Matched sessions:", len(comparison))
print("\nComparison preview:")
display(comparison.head(10))

print("\nLabel agreement:")
print(
    (comparison["human_label"] == comparison["predicted_label"])
    .value_counts()
)

Matched sessions: 51

Comparison preview:


,session_id,human_label,predicted_label,confidence
0,4f265faea72a,insufficient_evidence,insufficient_evidence,High
1,5b0c375bd6d9,reconnaissance,reconnaissance,High
2,d5b0e92a0bb7,reconnaissance,reconnaissance,High
3,27a1baf6ca1a,insufficient_evidence,insufficient_evidence,High
4,0c279854bc2f,insufficient_evidence,insufficient_evidence,High
5,830c5d707e19,insufficient_evidence,insufficient_evidence,High
6,30e256faef11,insufficient_evidence,insufficient_evidence,High
7,27e8c21a9be2,insufficient_evidence,insufficient_evidence,High
8,caaa842505d2,insufficient_evidence,insufficient_evidence,High
9,6e523048d6ac,insufficient_evidence,insufficient_evidence,High



Label agreement:
True    51
Name: count, dtype: int64


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

y_true = comparison["human_label"]
y_pred = comparison["predicted_label"]

labels = [
    "reconnaissance",
    "brute_force",
    "malicious_command_attempt",
    "insufficient_evidence"
]

print("ACCURACY")
print(accuracy_score(y_true, y_pred))

print("\nCLASSIFICATION REPORT")
print(classification_report(
    y_true,
    y_pred,
    labels=labels,
    zero_division=0
))

print("\nCONFUSION MATRIX")
cm = confusion_matrix(y_true, y_pred, labels=labels)

cm_df = pd.DataFrame(
    cm,
    index=labels,
    columns=labels
)

display(cm_df)

ACCURACY
1.0

CLASSIFICATION REPORT
                           precision    recall  f1-score   support

           reconnaissance       1.00      1.00      1.00         2
              brute_force       0.00      0.00      0.00         0
malicious_command_attempt       1.00      1.00      1.00        18
    insufficient_evidence       1.00      1.00      1.00        31

                 accuracy                           1.00        51
                macro avg       0.75      0.75      0.75        51
             weighted avg       1.00      1.00      1.00        51


CONFUSION MATRIX


,reconnaissance,brute_force,malicious_command_attempt,insufficient_evidence
reconnaissance,2,0,0,0
brute_force,0,0,0,0
malicious_command_attempt,0,0,18,0
insufficient_evidence,0,0,0,31


In [ ]:
# Save the joined evaluation dataset
comparison.to_csv("evaluation_results.csv", index=False)

# Check confidence distribution
print("CONFIDENCE DISTRIBUTION")
print(comparison["confidence"].value_counts(dropna=False))

# Save the confusion matrix
cm_df.to_csv("confusion_matrix.csv")

# Confirm saved files
print("\nFiles saved:")
print("evaluation_results.csv")
print("confusion_matrix.csv")

CONFIDENCE DISTRIBUTION
confidence
High    51
Name: count, dtype: int64

Files saved:
evaluation_results.csv
confusion_matrix.csv


In [ ]:
from google.colab import files

files.download("evaluation_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download("confusion_matrix.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

files.download("gemini_predictions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
%%writefile evaluate_predictions.py

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

DATA_DIR = "data"

human = pd.read_csv(f"{DATA_DIR}/human_review.csv")
predictions = pd.read_csv(f"{DATA_DIR}/gemini_predictions.csv")

comparison = pd.merge(
    human[["session_id", "human_label"]],
    predictions[["session_id", "predicted_label", "confidence"]],
    on="session_id",
    how="inner"
)

labels = [
    "reconnaissance",
    "brute_force",
    "malicious_command_attempt",
    "insufficient_evidence"
]

y_true = comparison["human_label"]
y_pred = comparison["predicted_label"]

print("Matched sessions:", len(comparison))
print("\nAccuracy:", accuracy_score(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=labels,
    zero_division=0
))

matrix = confusion_matrix(y_true, y_pred, labels=labels)

matrix_df = pd.DataFrame(
    matrix,
    index=labels,
    columns=labels
)

print("\nConfusion Matrix:")
print(matrix_df)

comparison.to_csv("evaluation_results.csv", index=False)
matrix_df.to_csv("confusion_matrix.csv")

print("\nEvaluation completed successfully.")

Overwriting evaluate_predictions.py


In [1]:
from google.colab import files
files.download("evaluate_predictions.py")


FileNotFoundError: Cannot find file: evaluate_predictions.py

In [3]:
%%writefile evaluate_predictions.py

import csv
from pathlib import Path

DATA_DIR = Path("data")

LABELS = [
    "reconnaissance",
    "brute_force",
    "malicious_command_attempt",
    "insufficient_evidence"
]

# Load human-reviewed labels
with open(DATA_DIR / "human_review.csv", newline="") as file:
    human_rows = list(csv.DictReader(file))

# Load Gemini predictions
with open(DATA_DIR / "gemini_predictions.csv", newline="") as file:
    prediction_rows = list(csv.DictReader(file))

# Match records using session IDs
human = {row["session_id"]: row["human_label"] for row in human_rows}
predictions = {row["session_id"]: row for row in prediction_rows}

matched_ids = sorted(set(human) & set(predictions))

comparison = []

for session_id in matched_ids:
    comparison.append({
        "session_id": session_id,
        "human_label": human[session_id],
        "predicted_label": predictions[session_id]["predicted_label"],
        "confidence": predictions[session_id]["confidence"]
    })

# Validate labels
for row in comparison:
    assert row["human_label"] in LABELS
    assert row["predicted_label"] in LABELS

# Build confusion matrix
matrix = {
    actual: {predicted: 0 for predicted in LABELS}
    for actual in LABELS
}

for row in comparison:
    matrix[row["human_label"]][row["predicted_label"]] += 1

total = len(comparison)
correct = sum(matrix[label][label] for label in LABELS)

print("Matched sessions:", total)
print("Accuracy:", correct / total if total else 0)

print("\nCLASSIFICATION REPORT")
print(f"{'Class':35} Precision  Recall  F1  Support")

all_metrics = []

for label in LABELS:
    tp = matrix[label][label]
    actual_count = sum(matrix[label].values())
    predicted_count = sum(matrix[actual][label] for actual in LABELS)

    precision = tp / predicted_count if predicted_count else 0
    recall = tp / actual_count if actual_count else 0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0
    )

    all_metrics.append((precision, recall, f1, actual_count))

    print(
        f"{label:35} {precision:.2f}      "
        f"{recall:.2f}    {f1:.2f}  {actual_count}"
    )

macro = [
    sum(metric[i] for metric in all_metrics) / len(LABELS)
    for i in range(3)
]

weighted = [
    sum(metric[i] * metric[3] for metric in all_metrics) / total
    if total else 0
    for i in range(3)
]

print("\nMacro average:", [round(x, 2) for x in macro])
print("Weighted average:", [round(x, 2) for x in weighted])

# Save evaluation results
with open(DATA_DIR / "evaluation_results.csv", "w", newline="") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=[
            "session_id",
            "human_label",
            "predicted_label",
            "confidence"
        ]
    )
    writer.writeheader()
    writer.writerows(comparison)

# Save confusion matrix
with open(DATA_DIR / "confusion_matrix.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["human_label"] + LABELS)

    for label in LABELS:
        writer.writerow(
            [label] + [matrix[label][predicted] for predicted in LABELS]
        )

print("\nEvaluation files saved successfully.")

Overwriting evaluate_predictions.py


In [ ]:
from google.colab import drive
drive.mount('/content/drive')